In [2]:
!pip install -q hazm tokenizers snowballstemmer

In [3]:
import json

from pathlib import Path


def load_documents(path, parsed_documents_path):
    path = Path(path)

    with path.open("r", encoding="utf-8-sig") as f:
      doc_file = json.load(f)

    documents = []

    for i, row in enumerate(doc_file, start=1):
        document = {
            "doc_id": f"document_{i:03d}",
            "title": str(row.get("title", "")).strip(),
            "abstract": str(row.get("abstract", "")).strip(),
            "author": str(row.get("authors", "")).strip(),
            "keywords": str(row.get("keywords", "")).strip(),
            "year": str(row.get("publication_year", "")).strip(),
            "degree": str(row.get("degree", "")).strip(),
            "document_type": str(row.get("document_type", "")).strip(),
            "advisor": str(row.get("advisor", "")).strip(),
            "url": str(row.get("url", "")).strip(),
            "doi": str(row.get("doi", "")).strip(),
        }

        if document["title"] or document["abstract"] or document["keywords"]:
            documents.append(document)

    save_documents(documents, parsed_documents_path)

    return documents


def save_documents(documents, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        json.dump(documents, f, ensure_ascii=False, indent=2)


def document_text(document):
    parts = [
        document.get("title", ""),
        document.get("abstract", ""),
        document.get("keywords", "")
    ]

    return " ".join(str(p) for p in parts).strip()

In [4]:
import html
import re
import unicodedata
import string

# Persian Character Standardization
char_translator = str.maketrans({
    "ي": "ی",
    "ك": "ک",
    "ؤ": "و",
    "أ": "ا",
    "ة": "ه",
    "ٱ": "ا",
    "ۀ": "ه",
    "إ": "ا",
})

# transform english and arabic digits to persian digits
digits_translator = str.maketrans(
    "0123456789٠١٢٣٤٥٦٧٨٩",
    "۰۱۲۳۴۵۶۷۸۹۰۱۲۳۴۵۶۷۸۹"
)

persian_punctuations = "،؛؟«»ـ…“”‘’"
all_punctuations = string.punctuation + persian_punctuations
punctuatuin_translator = str.maketrans('', '', all_punctuations)

# remove diacritics e.g. ُ ِ َ
diacritics_removal = re.compile(r"[\u064B-\u065F\u0670\u06D6-\u06ED]")

# remove urls
url_removal = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)

def normalize_text(text):
    text = html.unescape(text)

    # Unicode normalization
    text = unicodedata.normalize("NFC", text)

    text = text.casefold()

    # apply transforms
    text = text.translate(char_translator)
    text = text.translate(digits_translator)
    text = diacritics_removal.sub("", text)
    text = url_removal.sub(" ", text)

    text = text.translate(punctuatuin_translator)

    # remove half space
    text = text.replace("\u200c", " ").replace("\u200d", "").replace("\ufeff", "")

    return " ".join(text.split())

In [5]:
class RuleBasedPersianStemmer:
    def __init__(self):
        # self.suffixes = [
        #     "ی", "ای", "ها", "های", "هایی", "تر", "تری", "ترین", "گر", "گری",
        #     "ام", "ات", "اش", "یم", "ید", "ند", "مان", "تان", "شان", "ون",
        #     "هایمان", "هایتان", "هایشان", "ان", "ین", "هایش", "هایت", "هایم",
        #     "م", "ت", "ش",
        # ]

        self.suffixes = [
            "هایمان", "هایتان", "هایشان",
            "هایش", "هایت", "هایم",
            "هایی", "های", "ها",
            "ترین", "تری", "تر",
            "یان", "گان",
        ]

        self.derivational_suffixes = [
            "واره", "گانه", "انه", "مند",
            "ناک", "وار", "بان", "دان", "ار",
            "زار", "یت", "چه", "چی", "گر",
        ]

        self.mokassar_dict = {
            "کتب": "کتاب",
            "اشجار": "شجر",
            "علوم": "علم",
            "افراد": "فرد",
        }

        self.suffixes.extend(self.derivational_suffixes)

        self.suffixes.sort(key=len, reverse=True)

    def stem(self, word):
        if word in self.mokassar_dict:
            return self.mokassar_dict[word]

        for suffix in self.suffixes:
            if word.endswith(suffix):
                candidate = word[: -len(suffix)]

                if suffix == "ان" and candidate.endswith("ی") and len(candidate) > 2:
                    if candidate[:-1].endswith(("ا", "و")):
                        candidate = candidate[:-1]

                if len(candidate) >= 2:
                    word = candidate
                    break
        return word

In [6]:
from hazm import Lemmatizer

class HazmLemmaStep:
  def __init__(self):
    self.lemmatizer = Lemmatizer()

  def lemmatize(self, word):
    if self.lemmatizer is not None:
      result = self.lemmatizer.lemmatize(word)

      if result:
        return result.split("#")[0]

    return word

In [7]:
from hazm import stopwords_list

import snowballstemmer

stemmer = RuleBasedPersianStemmer()
lemmatizer = HazmLemmaStep()
# stemmer = snowballstemmer.stemmer('persian')


def space_tokenize(text):
    text = normalize_text(text)
    return text.split()


def transform_tokens(tokens, remove_stopwords):
    stopwords = set(stopwords_list())
    output = []

    for token in tokens:
      if remove_stopwords and token in stopwords:
        continue

      token = lemmatizer.lemmatize(token)
      # token = stemmer.stem(token)

      token = stemmer.stem(token)

      if token:
        output.append(token)

    return output

In [8]:
example_words = [
    "کتاب ها",
    "دانشجویان",
    "رفتند",
    "نوشتم",
    "علوم",
    "معلومات",
]

for word in example_words:
    normalized_word = normalize_text(word)
    print(
        f"{word:15} | stem: {stemmer.stem(normalized_word):15} "
        f"| lemma: {lemmatizer.lemmatize(normalized_word)}"
    )

کتاب ها         | stem: کتاب            | lemma: کتاب ها
دانشجویان       | stem: دانشجو          | lemma: دانشجویان
رفتند           | stem: رفتند           | lemma: رفت
نوشتم           | stem: نوشتم           | lemma: نوشت
علوم            | stem: علم             | lemma: علوم
معلومات         | stem: معلومات         | lemma: معلومات


In [9]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import WhitespaceSplit

def train_bpe_tokenizer(texts, vocab_size=16000, save_path="bpe_tokenizer.json"):
  tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

  tokenizer.pre_tokenizer = WhitespaceSplit()

  trainer = BpeTrainer(
      vocab_size=vocab_size,
      min_frequency=2,
      special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"]
  )

  tokenizer.train_from_iterator(texts, trainer=trainer, length=len(texts))
  tokenizer.save(str(save_path))
  return tokenizer


def bpe_tokenize(text, tokenizer):
  text = normalize_text(text)
  return tokenizer.encode(text).tokens


def bpe_tokenize_words(words, tokenizer):
  output = []
  for word in words:
    if word:
      output.extend(tokenizer.encode(word).tokens)
  return output


def longest_token(tokens):
  return max(tokens, key=len) if tokens else ""


def compare_tokenizers(documents, bpe_tokenizer):
    space_tokens_all = []
    bpe_tokens_all = []
    space_doc_lengths = []
    bpe_doc_lengths = []

    for document in documents:
        space_tokens = space_tokenize(document_text(document))
        bpe_tokens = bpe_tokenize(document_text(document), bpe_tokenizer)

        space_tokens_all.extend(space_tokens)
        bpe_tokens_all.extend(bpe_tokens)
        space_doc_lengths.append(len(space_tokens))
        bpe_doc_lengths.append(len(bpe_tokens))

    return {
        "space": {
            "corpus_vocabulary_size": len(set(space_tokens_all)),
            "average_tokens_per_document": round(sum(space_doc_lengths) / max(len(space_doc_lengths), 1), 2),
            "longest_token": longest_token(space_tokens_all),
            "longest_token_length": len(longest_token(space_tokens_all)),
            "model_vocabulary_size": None,
        },
        "bpe": {
            "corpus_vocabulary_size": len(set(bpe_tokens_all)),
            "average_tokens_per_document": round(sum(bpe_doc_lengths) / max(len(bpe_doc_lengths), 1), 2),
            "longest_token": longest_token(bpe_tokens_all),
            "longest_token_length": len(longest_token(bpe_tokens_all)),
            "model_vocabulary_size": len(bpe_tokenizer.get_vocab()),
        },
    }

In [10]:
import json
from pathlib import Path


def prepare_document_tokens(document, tokenizer_type, bpe_tokenizer=None, remove_stopwords=False):
    words = space_tokenize(document_text(document))
    words = transform_tokens(words, remove_stopwords=remove_stopwords)

    if tokenizer_type == "space":
        return words
    if tokenizer_type == "bpe":
        return bpe_tokenize(" ".join(words), bpe_tokenizer)


def build_index(documents, tokenizer_type="space", bpe_tokenizer=None, remove_stopwords=False):
    inverted_index = {}
    total_tokens = 0
    documents_data = {}

    for document in documents:
        doc_id = document["doc_id"]
        documents_data[doc_id] = {
            "title": document.get("title", ""),
            "author": document.get("author", ""),
            "year": document.get("year", ""),
        }

        tokens = prepare_document_tokens(
            document,
            tokenizer_type=tokenizer_type,
            bpe_tokenizer=bpe_tokenizer,
            remove_stopwords=remove_stopwords,
        )
        total_tokens += len(tokens)

        # Find positions of each term
        positions = {}
        for pos, term in enumerate(tokens):
            if term not in positions:
                positions[term] = []

            positions[term].append(pos)

        # Add terms to inverted index
        for term, term_positions in positions.items():
            if term not in inverted_index:
                inverted_index[term] = []

            inverted_index[term].append({
                "doc_id": doc_id,
                "tf": len(term_positions),
                "positions": term_positions
            })

    return {
        "meta": {
            "document_count": len(documents),
            "total_tokens": total_tokens,
            "vocabulary_size": len(inverted_index),
            "tokenizer_type": tokenizer_type,
        },
        "documents": documents_data,
        "inverted_index": inverted_index
    }


def save_index(index: dict, path: str | Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        json.dump(index, f, ensure_ascii=False, indent=2)


def index_statistics(index: dict, path: str | Path):
    postings = 0
    total_tf = 0

    for term, documents in index["inverted_index"].items():
        postings += len(documents)

        for document in documents:
            total_tf += document["tf"]

    size = Path(path).stat().st_size

    return {
        "document_count": index["meta"]["document_count"],
        "vocabulary_size": len(index["inverted_index"]),
        "number_of_postings": postings,
        "total_term_occurrences": total_tf,
        "index_size_bytes": size,
        "index_size_kb": round(size / 1024, 2)
    }

In [11]:
import re


class BooleanParser:
    def __init__(self, query):
        self.tokens = re.findall(
            r'"[^"]*"|\(|\)|\bAND\b|\bOR\b|\bNOT\b|[^\s()]+',
            query,
            re.IGNORECASE
        )

        self.pos = 0

        if not self.tokens:
            raise ValueError("Query is empty.")

    def peek(self):
        if self.pos < len(self.tokens):
            return self.tokens[self.pos]

        return None

    def consume(self):
        token = self.peek()

        if token is None:
            raise ValueError("Unexpected end of query.")

        self.pos += 1
        return token

    def match(self, operator):
        token = self.peek()
        if token and token.upper() == operator:
            self.consume()
            return True

        return False

    def parse(self):
        result = self.parse_or()

        if self.peek() is not None:
            raise ValueError(f"Unexpected token: {self.peek()}")

        return result

    # Lowest precedence
    def parse_or(self):
        result = self.parse_and()

        while self.match("OR"):
            result = ("OR", result, self.parse_and())

        return result

    # Medium precedence
    def parse_and(self):
        result = self.parse_not()

        while self.match("AND"):
            result = ("AND", result, self.parse_not())

        return result

    # Highest precedence
    def parse_not(self):
        if self.match("NOT"):
            return ("NOT", self.parse_not())

        return self.parse_primary()

    def parse_primary(self):
        token = self.peek()

        # Parentheses
        if token == "(":
            self.consume()
            node = self.parse_or()

            if self.peek() != ")":
                raise ValueError("Missing closing parenthesis.")

            self.consume()
            return node

        if token == ")":
            raise ValueError("Unexpected closing parenthesis.")

        if token.upper() in ("AND", "OR", "NOT"):
            raise ValueError(f"Operator {token} has no left operand.")

        # Phrase query
        words = []
        quoted = False
        while self.peek() is not None:
            token = self.peek()

            if token in ("(", ")") or token.upper() in ("AND", "OR", "NOT"):
                break

            token = self.consume()

            words.append(token)

        phrase = " ".join(words).strip()

        if not phrase.strip():
            raise ValueError("Empty phrase.")

        if quoted or len(words) > 1:
            return ("PHRASE", phrase)

        return ("TERM", phrase)

In [12]:
import json
from pathlib import Path


class SearchEngine:
    def __init__(self, index_path, bpe_tokenizer_path=None, tokenizer_type=None, documents_path=None, remove_stopwords=False):
        with Path(index_path).open("r", encoding="utf-8") as f:
            self.data = json.load(f)

        self.meta = self.data["meta"]
        self.documents = self.data["documents"]
        self.inverted = self.data["inverted_index"]

        self.original_documents = None
        if documents_path is not None:
            with Path(documents_path).open("r", encoding="utf-8") as file:
                rows = json.load(file)
            self.original_documents = {row["doc_id"]: row for row in rows}

        self.all_doc_ids = set(self.documents)

        self.tokenizer_type = self.meta["tokenizer_type"]
        self.remove_stopwords = remove_stopwords

        if self.tokenizer_type == "bpe":
          self.bpe_tokenizer = Tokenizer.from_file(str(bpe_tokenizer_path))

        # Create a dictionary for faster document lookup
        self.postings = {
            term: {p["doc_id"]: p for p in plist}
            for term, plist in self.inverted.items()
        }

    def process_query(self, text):
        words = space_tokenize(text)
        words = transform_tokens(words, remove_stopwords=self.remove_stopwords)

        if self.tokenizer_type == "space":
            return words
        return bpe_tokenize(" ".join(words), self.bpe_tokenizer)


    # search for an exact sequence of tokens
    def sequence_search(self, tokens):
        if not tokens:
            return set()

        posting_maps = [
            self.postings.get(token, {})
            for token in tokens
        ]

        # one missing token means no match
        if any(not pmap for pmap in posting_maps):
            return set()

        # find documents containing all tokens
        candidates = set(posting_maps[0])

        for pmap in posting_maps[1:]:
            candidates.intersection_update(pmap)

        results = set()

        for doc_id in candidates:
          positions = [
            set(pmap[doc_id]["positions"])
            for pmap in posting_maps
          ]

          for start in positions[0]:
            if all(start + i in positions[i] for i in range(1, len(tokens))):
              results.add(doc_id)
              break

        return results

    # Search for one term
    def term_search(self, term):
        tokens = self.process_query(term)

        if self.tokenizer_type == "word" and len(tokens) != 1:
            raise ValueError("Expected one word.")

        return self.sequence_search(tokens)

    # Search for a phrase
    def phrase_search(self, phrase):
      tokens = self.process_query(phrase)

      return self.sequence_search(tokens)

    def evaluate(self, node):
        kind = node[0]

        if kind == "TERM":
            return self.term_search(node[1])

        if kind == "PHRASE":
            return self.phrase_search(node[1])

        if kind == "NOT":
            return self.all_doc_ids - self.evaluate(node[1])

        if kind == "AND":
            return self.evaluate(node[1]) & self.evaluate(node[2])

        if kind == "OR":
            return self.evaluate(node[1]) | self.evaluate(node[2])

        raise ValueError(f"Unknown operator: {kind}")

    def boolean_search(self, query):
        ast = BooleanParser(query).parse()

        doc_ids = self.evaluate(ast)

        return self.format_results(doc_ids)

    def get_document(self, doc_id):
        return self.original_documents.get(doc_id)

    def format_results(self, doc_ids):
        results = []

        for doc_id in sorted(doc_ids):
            results.append({
                "doc_id": doc_id,
                "title": self.documents[doc_id].get("title", "")
            })

        return results


def prepare_results(results):
    if not results:
        return "(no matching documents)"

    return "\n".join(
        f"{r['doc_id']} | {r['title']}"
        for r in results
    )

In [13]:
def run_pipeline(
    dataset_path="IranDoc_Data.json",
    output_dir="search_engine_output",
    vocab_size=16000,
    remove_stopwords = False,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    documents = load_documents(
        dataset_path,
        parsed_documents_path=output_dir / "parsed_documents.json",
    )

    normalized_texts = [normalize_text(document_text(doc)) for doc in documents]

    bpe_path = output_dir / "bpe_tokenizer.json"

    bpe_tokenizer = train_bpe_tokenizer(
        normalized_texts,
        vocab_size=vocab_size,
        save_path=bpe_path,
    )

    tokenizer_comparison = compare_tokenizers(documents, bpe_tokenizer)

    index_paths = {}
    index_stats = {}

    for tokenizer_type in ("space", "bpe"):
        index = build_index(
            documents,
            tokenizer_type=tokenizer_type,
            bpe_tokenizer=bpe_tokenizer,
            remove_stopwords=remove_stopwords,
        )
        index_path = output_dir / f"index_{tokenizer_type}.json"
        save_index(index, index_path)

        key = f"{tokenizer_type}"
        index_paths[key] = str(index_path)
        index_stats[key] = index_statistics(index, index_path)

    documents_path = output_dir / "parsed_documents.json"
    search_engines = {}
    for key, index_path in index_paths.items():
        tokenizer_type = key.split("_", 1)[0]
        if tokenizer_type == "bpe":
            search_engines[key] = SearchEngine(
                index_path,
                bpe_tokenizer_path=bpe_path,
                tokenizer_type="bpe",
                documents_path=documents_path,
            )
        else:
            search_engines[key] = SearchEngine(
                index_path,
                tokenizer_type="space",
                documents_path=documents_path,
                remove_stopwords = remove_stopwords,
            )

    return {
        "documents": documents,
        "documents_path": str(documents_path),
        "bpe_tokenizer_path": str(bpe_path),
        "tokenizer_comparison": tokenizer_comparison,
        "index_paths": index_paths,
        "index_statistics": index_stats,
        "search_engines": search_engines,
    }


pipeline = run_pipeline(vocab_size = 2000)

In [15]:
def run_demo(boolean_engine, phrase_engine=None):
    if phrase_engine is None:
        phrase_engine = boolean_engine
    boolean_queries = [
        "بازیابی AND اطلاعات",
        "یادگیری AND ماشین",
        "بازیابی AND NOT وب",
        "(معنایی OR عصبی) AND بازیابی",
        "(دانشگاه OR پژوهشگاه) AND مقالات",
    ]
    phrase_queries = [
        'بازیابی اطلاعات',
        'هوش مصنوعی',
    ]

    print("BOOLEAN QUERIES")
    for query in boolean_queries:
        print("\nQuery:", query)
        print(prepare_results(boolean_engine.boolean_search(query)))

    print("\nPHRASE QUERIES")
    for query in phrase_queries:
        print("\nQuery:", query)
        print(prepare_results(phrase_engine.boolean_search(query)))


run_demo(pipeline["search_engines"]["space"])

BOOLEAN QUERIES

Query: بازیابی AND اطلاعات
document_048 | تاثیر هوش مصنوعی بر گردشگری ورزشی

Query: یادگیری AND ماشین
document_009 | تأثیر یک ابزار مبتنی بر هوش مصنوعی بر گرامر و تمایل زبان‌آموزان ایرانی برای برقراری ارتباط
document_011 | شناسایی عوامل موثر بر تجربه ی دیجیتال کارکنان در استفاده از ابزارهای هوش مصنوعی درسازمان
document_013 | طراحی مدل کاربردهای هوش مصنوعی در مدیریت منابع انسانی پروژه‌ها
document_016 | مدیریت داده‌های مراقبت بهداشتی و درمان بیماری عصبی با رویکرد ماشین ترکیبی در هوش مصنوعی
document_032 | پیش بینی بیماری‌ها‌ با‌ استفاده‌ از‌ هوش‌ مصنوعی‌
document_035 | تکینگی هوش مصنوعی و امکان پذیری آن
document_036 | سواد هوش مصنوعی معلمان زبان انگلیسی به عنوان زبان خارجی (EFL) در ایران و استفاده آن‌ها از هوش مصنوعی برای تدریس زبان قبل، حین و بعد از کلاس: مطالعه موردی مؤسسات زبان، مدارس و دانشگاه‌ها
document_037 | پیش بینی بیماری ها با استفاده از هوش مصنوعی
document_042 | مطالعه ظرفیت های هوش مصنوعی در برنامه سازی رادیویی
document_049 | ارائه چارچوب هوش رقابتیمبتنی بر هو

In [23]:
pipeline["index_statistics"]["space"] , pipeline["index_statistics"]["bpe"]

({'document_count': 100,
  'vocabulary_size': 3244,
  'number_of_postings': 16566,
  'total_term_occurrences': 37773,
  'index_size_bytes': 2318552,
  'index_size_kb': 2264.21},
 {'document_count': 100,
  'vocabulary_size': 1524,
  'number_of_postings': 19479,
  'total_term_occurrences': 46700,
  'index_size_bytes': 2697156,
  'index_size_kb': 2633.94})

In [24]:
pipeline["tokenizer_comparison"]

{'space': {'corpus_vocabulary_size': 3914,
  'average_tokens_per_document': 378.59,
  'longest_token': 'انگیزشدرونیانگیزشبیرونیو',
  'longest_token_length': 24,
  'model_vocabulary_size': None},
 'bpe': {'corpus_vocabulary_size': 1806,
  'average_tokens_per_document': 451.19,
  'longest_token': 'intelligence',
  'longest_token_length': 12,
  'model_vocabulary_size': 2000}}